# ISS_decoding of preprocessed files

This notebook guides you through the steps necessary for the decoding of ISS image data that have been preprocessed using our `ISS_preprocessing` module.

In this step, we make use of the starfish library to extract the information contained in our images. starfish is a Python library for processing images of image-based spatial transcriptomics. You can read more about it here: https://spacetx-starfish.readthedocs.io/en/latest/

In the following steps, we first format our images to a starfish-compatible format (SpaceTx), and provide some information about our experiment design. 
Once this is completed, we can proceed to the actual decoding of the data from the SpaceTx images.

For this notebook to work correctly, you should have a specific folder for each one of the regions you want to process, `/R1/`, `/R2/`, etc. Each region folder should contain the following subfolders tree: `/preprocessing/CycleX/4_retiled`. Under `/preprocessing/` you might still have other subfolders such as `/1_mipped/`, `/2_ome_tiffs/` and `/3_stitched/`, but these are irrelevant at this stage.

The important thing is that you have a `/preprocessing/CycleX/4_retiled/`, because the retiled images are the starting point of the decoding process.

## We start by importing the necessary modules

In [ ]:
import ISS_decoding.SpaceTx_format as STX
import ISS_decoding.decoding as DEC
import ISS_decoding.qc_metrics as QC
import pandas as pd
from pathlib import Path

## Format the images to SpaceTx format

The first thing to do before we can start the actual decoding is to transform our images (the resliced tiles) to the SpaceTx format.

To read more about the SpaceTx format, read the following: https://github.com/spacetx/sptx-format

### Parameters

`input_dir` = type: `str`. Path to the parent directory containing the preprocessed region folders (e.g., `/R1/`, `/R2/`, …).  
These region folders are automatically generated by the preprocessing module.  
The `input_dir` is referenced throughout this notebook.

`codebook_csv` = type: `str`. This is a file that associates a unique color sequence across ISS cycles to each gene. This file is a comma separated file with no header, in which the first column contains the gene name, while  columns 2 to 6 (in case of a 6 cycle experiment) contain numbers representing the expected positive DO_decorator in each cycle for that gene. 

`regions_to_process` = type:`list[int]` | None, default:None. A list of 1-based region indices defining which regions should be processed. If None → all detected regions are processed.

`output_dir_prefix` = type: `str` | None, default: `None`.  
Optional base directory where SpaceTx outputs should be written.

- If `output_dir_prefix` is `None`, SpaceTx outputs are written **inside each region directory under `input_dir`**, i.e.  
  `input_dir/R#/decoding/1_SpaceTX_format/`.

- If `output_dir_prefix` is set, SpaceTx outputs are written under:  
  `output_dir_prefix/R#/decoding/1_SpaceTX_format/`.

`pixel_to_um`  = type: `float`. 
Physical size of one pixel in microns (µm per pixel). This value determines the units of the spatial coordinates written to the experiment metadata and decoding outputs:

- `pixel_to_um = 1.0` (default) → coordinates are pixel-based.

- `pixel_to_um = 0.1625` (or microscope-specific value) → coordinates are in microns.


`channels` = type: `list`. The channels, in the order they were acquired in the microscope. Default = ["AF750", "Cy5", "Cy3", "AF488", "DAPI"]

`DO_decorators` =  type: `list` **This point can be tricky to understand.** This shows how you associate the numbers in the codebook (ie 1,2,3,4) to a specific list of colors (DO_decorator) . In our lab, 1,2,3,4,5 correspond to  ["AF750", "488", "Cy3",  "Cy5", "at425"]. 

Default =  ["AF750", "488", "Cy3",  "Cy5"]. Users who will follow our barcode design and readout schemes should not change this.

`nuclei_channel`  = type: `str`. This is the name of the channel that corresponds to your nuclei stained image. Default =  "DAPI".

`CARE` = type: `bool`.
If set to `True`, the function will use CARE-denoised retiled images as input.

- `CARE = False` (default):
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/`

- `CARE = True`:
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/CARE/`
This folder must already exist and should contain the outputs produced by the ISS_CARE preprocessing step.
This parameter does not run CARE itself. It only controls which image directory is used as input when building the experiment and codebook files.
Users should set `CARE = True` only after CARE denoising has been successfully completed for the corresponding regions and cycles.






In [ ]:
input_dir = '/path/to/regions/'
codebook_csv = '/path/to/codebook/'

In [ ]:
STX.make_spacetx_format(
    input_dir,
    codebook_csv,
    regions_to_process = None,    # or for example [2]
    output_dir_prefix = None,     # or '/path/to/preferred/output/dir'
    pixel_to_um = 1,
    channels = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"],
    DO_decorators = ["AF750", "AF488", "Cy3", "Cy5", "At425"],
    nuclei_channel = "DAPI",
    CARE = True                  # True if using CARE denoised images
    )


## Decode the SpaceTx formatted data

At this stage, after running the above code block, you should have successfully transformed your preprocessed tiles into the SpaceTx format. Each `/decoding/1_SpaceTX_format/` directory should now contain a file called `experiment.json`.

To decode the images, we now use the `process_experiment` function.

In brief, the function takes each individual SpaceTx image tile, identifies the same spot across cycles, infers a colour sequence for each spot across the different cycles, and matches the extracted sequence to the provided `codebook`.

In detail, this process is more complex and involves several steps within the `process_experiment` function. We recommend consulting the manual for a full explanation of what happens under the hood.



### Note on spatial units

In the decoding module, spatial units are determined by the `pixel_to_um` parameter used when generating the SpaceTx experiment:

- If `pixel_to_um = 1`, decoded coordinates are in **pixels**
- If `pixel_to_um` is set to a microscope-specific value (e.g. `0.1625`), decoded coordinates are in **microns**

This parameter affects **decoding coordinate units only**. 


### Changing decoding parameters

Modifying the input parameters of the `process_experiment` function allows exploration of different decoding settings to optimize results for a specific experiment.

#### Parameters

`input_dir` = type: `str`  
Path to the directory containing the preprocessed region folders (`/R1/`, `/R2/`, etc.). This is the same directory used in preprocessing.


`regions_to_process` = type: `list[int] | None`, default: `None`  
A list of 1-based region indices to process.  
If `None`, all detected regions are processed.


`output_dir_prefix` = type: `str | None`, default: `None`  
Optional base directory where decoding outputs should be written.

- If `None`, outputs are written inside each region:
  ```
  input_dir/R#/decoding/2_decoded/
  ```

- If set, outputs are written to:
  ```
  output_dir_prefix/R#/decoding/2_decoded/
  ```

- If `dense=True`, outputs are written to:
  ```
  .../decoding/2_decoded_dense/
  ```



`pixel_to_um` = type: `float`, default: `1.0`  
Spatial scaling used for the SpaceTx experiment coordinates and therefore for decoded spot coordinates.

- `pixel_to_um = 1.0` → decoded coordinates are **pixel-based**
- `pixel_to_um = 0.1625` (or microscope-specific value) → decoded coordinates are **in microns**


`register` = type: `bool`  
Set to `True` to perform image registration.  
Although preprocessing usually handles alignment, this step can improve results in some cases.


`register_dapi` = type: `bool`  
If `True`, registration is based on nuclei (DAPI) staining.  
If `False`, registration uses a pseudo-anchor image.  
Only used if `register=True`.


`masking_radius` = type: `int`  
Radius of the WhiteTophat filter used for spot enhancement.  
Typical values range from 7 to 15. We recommend starting with 7.


`normalization_method` = type: `str`  
Options: `'MH'` or `'CPTZ'`.

- `'MH'` (Match Histograms) is recommended for ISS data  
- `'CPTZ'` is an alternative normalization method


`decode_mode` = type: `str`  
Options:
- `'PRMC'` = Per Round Max Channel (recommended)
- `'MD'` = Metric Distance

Refer to the manual for differences between methods.


`dense` = type: `bool`  
If `True`, enables **dense decoding mode**.

- `False` (default): spots are detected on a pseudo-anchor image (faster, good for sparse data)
- `True`: spots are detected per channel (slower, but better for dense data)

Dense mode can be ~5× slower.


`spot_detection_mode` = type: `str`  
Options: `'starfish'` or `'spotiflow'`  
(**Note: Spotiflow currently not working**)


`int_threshold` = type: `float`  
Minimum intensity threshold for spot detection (Starfish mode).

- Lower values → more detected spots (including noise)
- Higher values → fewer, higher-confidence spots

`sigma_vals` = type: `list`  
Corresponds to `[min_sigma, max_sigma, num_sigma]` for blob detection.  
See:
https://spacetx-starfish.readthedocs.io/en/latest/api/spots/index.html#spot-finding

Default values are tuned for common ISS setups (Leica/Zeiss, 20x/40x).

`prob_threshold` = type: `float | None`  
Confidence threshold for Spotiflow predictions (0–1).  
If `None`, an optimal value is used.

#### Output

The decoding step produces `.csv` files containing:

- Spot identity (gene)
- Spot location (X, Y coordinates)
- Quality metrics (e.g. intensity ratios)

Additionally, each decoded CSV records the coordinate system used:

- `coordinate_units` → `"pixels"` or `"microns"`
- `coordinate_pixel_to_um` → scaling factor used

This ensures that downstream analyses can correctly interpret spatial coordinates.

In [ ]:
DEC.process_experiment(
    input_dir, 
    regions_to_process = None,             # or for example [1,4,5]
    output_dir_prefix = None,              # or '/path/to/preferred/output/dir'
    pixel_to_um=1,
    register = False,
    register_dapi = False,
    masking_radius = 7,
    normalization_method = 'MH',
    decode_mode = 'PRMC',
    dense=False,
    spot_detection_mode='starfish',
    int_threshold = 0.002, 
    sigma_vals = [1, 10, 30] # min, max and number
    )

# Explore the decoded data

Now we should have as an output a big table (saved in csv) containing the information of our raw decoded spots. The main columns are: 

1. The location of the decoded spot (columns “xc” and “yc”) and 

2. the identity of every spot (column: “target”). 

Similarly to what happens in NGS experiments, there "raw" reads migh have variable quality and one key step before performing downstream analyses is to assess this quality and filter the data if necessary (**spoiler alert: it is always necessary to do some level of filtering**)

#### Processing Reads for Individual Regions

We will now process each region individually (`R1`, `R2`, etc.).
For each region, we must assess the quality of its reads separately to ensure accurate results for that specific region.

**Step 1:** Read the combined CSV file for a single region.

- **Inputs:**
  - `region` (string): Name of the region, e.g., `'R1'`, `'R2'`.
  - `dense` (boolean):  
    - `True` → read from `2_decoded_dense/{region}_decoded.csv`  
    - `False` → read from `2_decoded/{region}_decoded.csv`
- **Path construction:**  
  The file path should be built automatically from `input_dir`, `region`, and `dense`.



In [ ]:
region = 'R1'
dense = False

read_file = (
    Path(input_dir)
    / region
    / 'decoding'
    / f"2_decoded{'_dense' if dense else ''}"
    / f"{region}_decoded.csv"
)

reads = pd.read_csv(read_file)


The number of extracted raw reads for this section is:

In [ ]:
len(reads)

## Understanding read quality

To have an idea of how the read quality looks like, we can generate a violin plot of the qualities per cycle using the `quality_per_cycle` function. The function gets the data from the `reads` table above. The number of cycles is infered from the `reads` table. 

You can have a look at the manual to understand how the quality is calculated. Summing up, 1 is the theoretical maximum and 0.25 the theoretical minimum in a 4 colour setting. The more your violin plots are shifted towards 1, the better.

In [ ]:
QC.quality_per_cycle(reads)

Another useful plot to generate is to see how quality score metrics reflect whether a read has a match on the codebook ("assigned") or not ("non assigned"). We plot the `quality_minimum` and `quality_mean` against the assignment/non assignment to understand which is the best strategy to filter our data. Ideally we should infer from this plot a good quality threshold for filtering our data. 

In [ ]:
QC.compare_scores(reads,score1='quality_mean',score2='quality_minimum',hue='assigned',kind='hist',color='#3266a8')


The following command does the same, but only on a single score.

In [ ]:
QC.plot_scores(reads, on='quality_mean', hue='assigned', log_scale=False)


## Filtering the data

There are different strategies to filter the data, depending on what the above plots show. You can refer to the manual for more specific example.

A general common sense criterion is to filter out all the reads whose `quality_minimum` is < 0.5
This discards all the reads that show poor quality in **at least one cycle**. This is quite a conservative criterion, but as a rule of thumb is a good one.


Other methods for filtering are also described in the relevant part of the manual.

In [ ]:
reads_filt = QC.filter_reads(
    reads,
    min_quality_minimum=0.5,
    save_file=True,
    source_file=read_file
)

The number of filtered reads for this section is:

In [ ]:
len(reads_filt)

## Other useful functions

`quality_per_gene`: this function plots the quality of the reads assigned to each gene. Since every gene has an associated sequence of colors across cycles, we might have a big quality bias between different genes arising just because of the colour sequence (ie. if a channel has always a bad signal/noise, genes with many cycles in that channels will always have lower quality).

In [ ]:
QC.quality_per_gene(reads,on='quality_mean',gene_name='target')

`plot_frequencies`: allows us to plot the relative abundance of the decoded dots for each gene. This is useful to identify potential decoding artefacts. For example, a gene that looks 1000 times more abundant than the second more represented gene, should raise an eyebrow.

In [ ]:
QC.plot_frequencies(reads,on='target')

# Plot expression data (currently deprecated in favor of TissUUmaps)

After quality-filtering, a sensible thing to do is to plot the gene expression decoded from your tissue. 

Parameters `xcolumn` and `ycolumn` define the XY positions of your spots in the `reads_filt` dataframe. 

`key` points to the column containing the gene identity of your spots in the `reads_filt` dataframe.

`genes` allows to plot a single gene per plot (=`individual`), or to plots all genes in a single plot (=`all`)

`size`= sets the dot size

`background`= sets the background color	

`title_color`= sets the color of the title for each plot.

`colorcode`= sets the color of the expression dots

`figuresize`=(10,10) sets the size of the plot

`save`= defaults to `None`. If `True` saves the plot in the speficied `format`

`format`= saves the plot to a specific format (ie `pdf`)


In [ ]:
QC.plot_expression(reads,key='target',colorcode=['red'],xcolumn='xc',ycolumn='yc',genes='individual',size=8,background='black',title_color='white',figuresize=(10,10),save=None,format='pdf')
